# Magnetic Resonance Imaging II

## Exercise 1 - 2D Gradient-echo sequence with simulation via MRZero and reconstruction

**Prof. Dr. Markus Zimmermann**

**Mohamed Wassim Bensalem**

## Software Imports

Description here

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch

## System and Sequence Parameters

Description here

In [ ]:
seq_filename = "gre.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128

case = ["T1-weighted", "T2-weighted", "PD-weighted"][1]

if case == "T1-weighted":
    flip_angle_deg = 90
    tr = 250e-3
    te = 5e-3
elif case == "T2-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 50e-3
elif case == "PD-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 5e-3

torch.set_default_device("cpu")

## System

Description here

In [ ]:
# Your solution here

system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

## RF-Pulse

Description here

In [ ]:
# Your solution here

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

## Frequency- and Phase-Encoding Gradients

Description here

In [ ]:
# Your solution here

delta_kx = 1 / fov_x
delta_ky = 1 / fov_y

phase_areas = np.linspace(-n_y / 2, n_y / 2, num=n_y, endpoint=False) * delta_ky

gx = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=3.2e-3,
    system=system,
)

gx_pre = pp.make_trapezoid(
    channel="x",
    area=-gx.area / 2,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=gx.flat_time,
    delay=gx.rise_time,
    system=system,
)

## Spoiler Gradients

Description here

In [ ]:
# Your solution here

gx_spoil = pp.make_trapezoid(
    channel="x",
    area=4 * gx.area,
    system=system,
)

## Timing

Description here

In [ ]:
# Your solution here

te_delay = (
    te
    - (pp.calc_duration(rf) - pp.calc_rf_center(rf)[0])
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx) / 2
)

tr_delay = (
    tr
    - pp.calc_duration(rf)
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx)
    - pp.calc_duration(gx_spoil)
    - te_delay
)

## Sequence

Description here

In [ ]:
# Your solution here

seq = pp.Sequence(system)

EPS = 1e-12

te_delay = np.ceil((te_delay - EPS) / seq.grad_raster_time) * seq.grad_raster_time
tr_delay = np.ceil((tr_delay - EPS) / seq.grad_raster_time) * seq.grad_raster_time

assert np.all(te_delay >= 0)
assert np.all(tr_delay >= 0)

for phase_area in phase_areas:

    seq.add_block(rf)

    gy_pre = pp.make_trapezoid(
        channel="y",
        area=phase_area,
        duration=pp.calc_duration(gx_pre),
        system=system,
    )

    seq.add_block(gx_pre, gy_pre)

    seq.add_block(pp.make_delay(te_delay))

    seq.add_block(gx, adc)

    seq.add_block(gx_spoil)

    seq.add_block(pp.make_delay(tr_delay))

## Check

Description here

In [ ]:
# Your solution here

ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

## Plot

Description here

In [ ]:
# Your solution here

seq.plot(time_range=(0.0, tr - tr_delay + te_delay))

## Write sequence

Description here

In [ ]:
# Your solution here

seq.write(seq_filename)

## Create Phantom

Description here

In [ ]:
# Your solution here

phantom = mr0.util.load_phantom(size=(n_x, n_y))

## Plot Phantom

Description here

In [ ]:
# Your solution here

phantom.plot()

## Simulation

Description here

In [ ]:
# Your solution here

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Reconstruction

Description here

In [ ]:
# Your solution here

signal = signal.numpy()

signal = signal.reshape((n_x, n_y))

image = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(signal)))

## Display results

Description here

In [ ]:
# Your solution here

plt.figure()
plt.imshow(np.abs(image), cmap="gray")
plt.title("Reconstructed GRE Image")
plt.show()